In [5]:
# ---------------------------------------
# 
# read in era5 data and move to aws
# 
# [ *** OUTDATED: check out 
#  `retrieve-data-gcp.ipynb`]
# 
# ---------------------------------------
import os
import itertools
import shutil

import cdsapi
import s3fs
import tempfile
import xarray as xr
from pathlib import Path

# --- file destination
dst = 's3://carbonplan-carbon-removal/era5/preprocessed_data/'

# --- set the file system object
fs = s3fs.S3FileSystem(anon=False)

# --- set the working dir (for this script)
input_dir = Path("/home/tykukla/ew-workflows/era5/download_scripts")

In [6]:
# --- define the vars we want to collect
dataset = "reanalysis-era5-land"

# [ for a varlist: https://cds.climate.copernicus.eu/datasets/reanalysis-era5-land?tab=download ]
era5vars = ["2m_temperature",
        "skin_reservoir_content",
        "volumetric_soil_water_layer_1",
        "volumetric_soil_water_layer_2",
        "volumetric_soil_water_layer_3",
        "volumetric_soil_water_layer_4",
        "evaporation_from_vegetation_transpiration",
        "potential_evaporation",
        "runoff",
        "total_evaporation",
        "total_precipitation",
        "geopotential",
        "land_sea_mask",
        "soil_type"
        ]

yrs = ["2011", "2012", "2013", "2014", "2015", "2016", "2017", "2018", "2019", "2020"]
mos = ["01", "02", "03", "04", "05", "06", "07", "08", "09", "10", "11", "12"]
days = [
        "01", "02", "03",
        "04", "05", "06",
        "07", "08", "09",
        "10", "11", "12",
        "13", "14", "15",
        "16", "17", "18",
        "19", "20", "21",
        "22", "23", "24",
        "25", "26", "27",
        "28", "29", "30",
        "31"
    ]
times = [
        "00:00", "01:00", "02:00",
        "03:00", "04:00", "05:00",
        "06:00", "07:00", "08:00",
        "09:00", "10:00", "11:00",
        "12:00", "13:00", "14:00",
        "15:00", "16:00", "17:00",
        "18:00", "19:00", "20:00",
        "21:00", "22:00", "23:00"
    ]

area = [50, -130, 20, -70]  # [maxlat, minlon, minlat, maxlon]

In [7]:
def create_era5_request(
    tvar: str,
    yr: str,
    month: str,
    day: list,
    time: list,
    area: list,
    dat_format: str="netcdf",
    dload_format: str="unarchived",
):
    '''
    Create the API request
    '''
    request = {
        "variable": [tvar],
        "year": [yr],
        "month": [month],
        "day": day,
        "time": time,
        "data_format": "netcdf",
        "download_format": "unarchived",
        "area": area
    }

    return request


In [8]:
# --- loop through vars and timesteps
# track whether we've written first chunk
first = True

for era5var, yr, mo in itertools.product(era5vars, yrs, mos):
    print(f"{era5var=} {yr=} {mo=}")
    # create filename 
    out_filename = f"{era5var}-{yr}-{mo}.nc"

    # only run this loop if the file doesn't exist yet
    if fs.exists(os.path.join(dst, out_filename)):
        print(f"skipping existing file: {out_filename}")
        continue

    # create request
    request = create_era5_request(
                                    tvar=era5var,
                                    yr=yr,
                                    month=mo,
                                    day=days,
                                    time=times,
                                    area=area,
                                    dat_format="netcdf",
                                    dload_format="unarchived",
                                )
    # submit request 
    client = cdsapi.Client()
    client.retrieve(dataset, request).download()

    # find the .nc file that was just added
    nc_files = list(input_dir.glob("*.nc"))
    if len(nc_files) != 1:
        raise ValueError(f"Expected one .nc file, found {len(nc_files)}")

    # open the dataset
    ds = xr.open_dataset(nc_files[0])
    # --- find append_dim
    # Option 1: Try known options
    for dim_candidate in ["valid_time", "time"]:
        if dim_candidate in ds.dims:
            append_dim = dim_candidate
            break

    # Option 2: Fallback - find 1D datetime-like coordinate
    if append_dim is None:
        for dim in ds.dims:
            coord = ds.coords.get(dim)
            if coord is not None and coord.ndim == 1 and np.issubdtype(coord.dtype, np.datetime64):
                append_dim = dim
                break
    if append_dim is None:
        raise ValueError(f"No appendable time-like dimension found in {nc_path}")


    # # Write or append to S3 Zarr
    # ds.to_zarr(
    #     store=dst,
    #     mode="w" if first else "a",
    #     append_dim=append_dim if not first else None,
    #     consolidated=True if first else False,
    #     storage_options={"anon": False},
    #     compute=True,
    # )

    # Save to S3 as NetCDF
    out_path = dst
    # Save to temporary file first
    with tempfile.TemporaryDirectory() as tmpdir:
        tmp_path = Path(tmpdir) / out_filename
        ds.to_netcdf(tmp_path)
        # move to s3
        fs.put(str(tmp_path), out_path, recursive=False)

    
    print(f"✅ Wrote {nc_files[0]}")

    # Delete file after success
    nc_files[0].unlink()
    print(f"🗑️ Removed local file: {nc_files[0]}")

    first = False


era5var='2m_temperature' yr='2011' mo='01'
skipping existing file: 2m_temperature-2011-01.nc
era5var='2m_temperature' yr='2011' mo='02'
skipping existing file: 2m_temperature-2011-02.nc
era5var='2m_temperature' yr='2011' mo='03'
skipping existing file: 2m_temperature-2011-03.nc
era5var='2m_temperature' yr='2011' mo='04'
skipping existing file: 2m_temperature-2011-04.nc
era5var='2m_temperature' yr='2011' mo='05'
skipping existing file: 2m_temperature-2011-05.nc
era5var='2m_temperature' yr='2011' mo='06'
skipping existing file: 2m_temperature-2011-06.nc
era5var='2m_temperature' yr='2011' mo='07'
skipping existing file: 2m_temperature-2011-07.nc
era5var='2m_temperature' yr='2011' mo='08'
skipping existing file: 2m_temperature-2011-08.nc
era5var='2m_temperature' yr='2011' mo='09'
skipping existing file: 2m_temperature-2011-09.nc
era5var='2m_temperature' yr='2011' mo='10'
skipping existing file: 2m_temperature-2011-10.nc
era5var='2m_temperature' yr='2011' mo='11'
skipping existing file: 2m_

In [ ]:
# --------------------------------------------